<a href="https://colab.research.google.com/github/prachimishraa/GenAI/blob/main/GenAI_Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf chromadb groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.2 MB/s eta 0:00:00


In [ ]:
import os
import fitz
import chromadb
from groq import Groq
from google.colab import userdata

try:
  groq_api_key = userdata.get("GROQ_API_KEY")
except Exception:
  import getpass
  groq_api_key = getpass.getpass("Enter your groq api key: ")

client = Groq(api_key=groq_api_key)
db = chromadb.Client()
collection = db.create_collection(name="handbook_rag", get_or_create=True)

PDF_FILE_PATH = "/content/HandBook_b9c6eSTUDENT HANDBOOK 2025 (1).pdf"

doc = fitz.open(PDF_FILE_PATH)
documents, metadatas, ids = [], [], []
chunk_counter = 0

for page_index, page in enumerate(doc):
  text = page.get_text()
  for i in range (0, len(text), 400):
    chunk = text[i:i + 500].strip()
    if chunk:
      documents.append(chunk)
      metadatas.append({"page": page_index + 1})
      ids.append(str(chunk_counter))
      chunk_counter += 1
collection.add(documents=documents, metadatas=metadatas, ids=ids)

def ask_question(question: str):
    query_results = collection.query(query_texts=[question], n_results=3)

    retrieved_texts = query_results['documents'][0]
    retrieved_metas = query_results['metadatas'][0]

    page_numbers = sorted(list(set([f"Page {meta['page']}" for meta in retrieved_metas])))
    context_text = "\n\n".join(retrieved_texts)

    prompt = f"""
    You are an AI assistant for university students. Answer the student's question using ONLY the context provided below.
    If the answer is not mentioned in the context, explicitly reply: "I cannot find this information in the student handbook."

    Context:
    {context_text}

    Question: {question}

    Answer:
    """

    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        temperature=0.0
    )

    generated_answer = chat_completion.choices[0].message.content
    preview_context = context_text[:200].replace('\n', ' ')

    print("=" * 80)
    print(f"Student Question:\n  └─ {question}\n")
    print(f"Retrieved Context:\n  └─ \"{preview_context}...\"\n")
    print(f"Generated Answer:\n  └─ {generated_answer.strip()}\n")
    print(f"Source/Page Number:\n  └─ {', '.join(page_numbers)}")
    print("=" * 80)

user_question = input("Enter your question: ")
if user_question.strip():
    ask_question(user_question)

Enter your groq api key: ··········
